# Argus — Dataset Creation (Face Crops for CNN)

This notebook builds the dataset for the fourth model family: a small CNN that classifies
drowsiness directly from a **cropped RGB face image**, with no hand-engineered geometric features
at all (no EAR/MAR, no blendshapes, no head pose). Like the flat per-frame dataset, this is a
single-image-in, single-label-out task — no sequence, no buffer.

**Why a separate, lighter face detector instead of reusing `FaceLandmarker`.** Every other
notebook uses MediaPipe's `FaceLandmarker`, which is comparatively heavy (478 3D landmarks + 52
blendshapes + a transformation matrix) because the geometric-feature pipeline needs all of that.
This notebook only needs a face *bounding box* to crop — so it uses MediaPipe's dedicated
**Face Detector** task (`BlazeFace`, short-range model) instead, which is smaller and faster
since it skips landmark/blendshape estimation entirely. This is a genuinely different MediaPipe
model bundle, downloaded separately below, not a lighter-weight mode of `FaceLandmarker`.

**Output:** cropped face images written as individual `.jpg` files under
`dataset_processed/face_crops/`, indexed by `dataset_processed/face_crops_index.csv`
(subject/level/parent_video/frame_idx/image_path) — a CSV of paths + labels rather than the
`ImageDataGenerator`-style one-subfolder-per-class layout, so that
`07_cnn_training.ipynb` can do the same subject-grouped train/test split every other notebook
uses (a directory-per-class layout has no natural place to carry subject IDs for that).

**Label mapping** matches `02_dataset_creation_flat.ipynb`'s convention. **Sampling rate:** this
notebook samples at `sampling_fps = 5`, capped at `MAX_FRAMES_PER_CLIP = 100` crops per clip
(≈ the first 20 s of a clip) — see Pipeline Configuration Constants below. That is denser than an
earlier version of this notebook, which sampled at 1 FPS specifically to avoid near-duplicate
crops. The CNN / CNN+LSTM path turned out to be starved for training volume on this project's
small subject pool, so the priority shifted: produce substantially more crops per clip and accept
that consecutive crops at 5 FPS are fairly similar. That redundancy is a real cost being traded
for volume, not a free gain — but more (even partly redundant) labelled crops still helps a
subject-scarce CNN more than it hurts. Extraction is parallelised across worker processes, so the
added wall-clock time is tolerable.


In [ ]:
# Drowsiness class labels. The raw clips arrive already binary -- each filename encodes
#   level_1 = "Not Drowsy"  (fully-alert + low-vigilant footage)
#   level_2 = "Drowsy"
# map_level (Pipeline Configuration Constants cell below) only validates that number; there
# is no class-remapping step anywhere in this pipeline. Keep this pair identical across
# 01_dataset_creation_lstm / 02_dataset_creation_flat / 06_dataset_creation_face_crops /
# 09_dataset_creation_cnn_lstm.
CLASS_NAMES = ["Not Drowsy", "Drowsy"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")


## Google Drive Connection & Project Setup


In [ ]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive')

project_folder = "/content/drive/MyDrive/Argus"
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
raw_videos_folder = f"{dataset_folder}/raw_videos_binary"
processed_folder = f"{dataset_folder}/dataset_processed"
face_crops_folder = f"{processed_folder}/face_crops"

if not os.path.exists(raw_videos_folder):
    raise Exception(f"The target directory could not be initialized: '{raw_videos_folder}'.")

for folder in [models_folder, processed_folder, face_crops_folder]:
    os.makedirs(folder, exist_ok=True)

video_files = glob.glob(os.path.join(raw_videos_folder, "**/*.mp4"), recursive=True)
if len(video_files) == 0:
    raise Exception(f"No video clips found in the specified directory: '{raw_videos_folder}'.")

print(f"Google Drive successfully mounted! Base project directory: {project_folder}")
print("Project folder structure successfully created and verified.")
print(f"[SUCCESS] Raw video directory validated: Found {len(video_files)} clip(s) ready for processing.")


## MediaPipe Face Detector Setup

Downloads the pretrained `BlazeFace` short-range bundle (bounding-box-only face detection — no
landmarks/blendshapes). This is a different, smaller model file from `FaceLandmarker`'s bundle
used in the other dataset-creation notebooks.


In [ ]:
!pip install mediapipe opencv-python

import urllib.request
import os
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

FACE_DETECTOR_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
face_detector_model_path = os.path.join(models_folder, "blaze_face_short_range.tflite")

if not os.path.exists(face_detector_model_path):
    print(f"Downloading Face Detector model to {face_detector_model_path}...")
    urllib.request.urlretrieve(FACE_DETECTOR_MODEL_URL, face_detector_model_path)
    print("✅ Download complete.")
else:
    print(f"✅ Face Detector model already present at {face_detector_model_path}.")

# Quick smoke test only -- confirms the downloaded model file actually loads before the
# expensive parallel extraction loop runs. The real FaceDetector instances used during
# extraction are built fresh per-video, per-worker (see FaceCropExtractionPipeline.process_video
# below), from face_detector_model_path directly, not from anything constructed here.
_smoke_test_options = vision.FaceDetectorOptions(
    base_options=mp_python.BaseOptions(model_asset_path=face_detector_model_path),
    running_mode=vision.RunningMode.IMAGE,
)
_smoke_test_detector = vision.FaceDetector.create_from_options(_smoke_test_options)
_smoke_test_detector.close()
print("✅ Face Detector model loads and initializes correctly.")


## Pipeline Configuration Constants

The label-mapping convention (`map_level`) matches `01_dataset_creation_lstm.ipynb` /
`02_dataset_creation_flat.ipynb` exactly — a `{1, 2}` validating pass-through (the raw filenames
already carry the final `level_1` / `level_2` label). Keep it in sync with those two notebooks.

`sampling_fps` still does not need to match those notebooks — the row/window semantics differ —
but it is now set to `5` (raised from an earlier `1`). The CNN and CNN+LSTM models consume raw
crops directly and were data-starved on this project's small subject pool, so this notebook now
prioritises volume: `sampling_fps = 5` with `MAX_FRAMES_PER_CLIP = 100` yields roughly the first
20 seconds of each clip as crops — many more per clip than the old 1-FPS / 20-cap setting. The
tradeoff is real and worth stating plainly: consecutive crops at 5 FPS are fairly similar, so a
meaningful fraction of the added crops are partial near-duplicates rather than fully independent
examples. It is accepted here because more (even partly redundant) labelled crops still helps a
subject-scarce CNN more than it hurts, and because extraction is parallelised across worker
processes so the added wall-clock time is tolerable. `MAX_FRAMES_PER_CLIP` still caps how many
crops any single clip can contribute, regardless of its duration. Two more constants specific to
face cropping: a confidence threshold for keeping a detection, and a margin added around the raw
bounding box so crops include a bit of context (forehead/chin/ears) rather than being cut exactly
at the eyes/mouth.


In [ ]:
# --- Pipeline Configuration Constants ---
sampling_fps = 5                        # Raised from an earlier 1 FPS: the CNN / CNN+LSTM path
                                        # consumes raw crops and was starved for training volume
                                        # on a small subject pool, so this now favours more crops
                                        # per clip over minimising near-duplicate frames. The
                                        # other dataset-creation notebooks use ~10 FPS.
MAX_FRAMES_PER_CLIP = 100               # Hard cap on crops per clip, independent of duration --
                                        # at 5 FPS this is ~the first 20s of a clip. A simple
                                        # ceiling, not an adaptive/similarity-based sampler.
min_detection_confidence = 0.5         # Frames with no detection above this are dropped
bbox_margin_frac = 0.25                # Expand the raw bounding box by this fraction on each side
jpeg_quality = 90

# --- Raw level (from filenames) -> validated class label ---
# Filenames already encode the final class (level_1 = Not Drowsy, level_2 = Drowsy), so
# map_level does no remapping -- it's a validation pass-through, kept as a function so a
# malformed level fails loudly at one place. Keep identical to the copies in
# 01_dataset_creation_lstm.ipynb / 02_dataset_creation_flat.ipynb.
def map_level(raw_level: int) -> int:
    if raw_level not in (1, 2):
        raise ValueError(f"Unexpected level {raw_level} in filename -- expected 1 (Not Drowsy) or 2 (Drowsy).")
    return raw_level

print(f"✅ Pipeline constants initialized. Sampling: {sampling_fps} FPS (max {MAX_FRAMES_PER_CLIP} crops/clip), margin: {bbox_margin_frac:.0%}")
print(f"   Labels: {NUM_CLASSES}-class {CLASS_NAMES}")


## Face Crop Extraction Pipeline

For each video: run the Face Detector at `sampling_fps`, and for every frame with a
confident detection, expand its bounding box by `bbox_margin_frac`, clip to the frame bounds, and
crop. Frames with no confident detection are dropped (same "invalid frame" concept as the other
notebooks, just detector-confidence-based here instead of pose-validity-based).


In [ ]:
import cv2
import numpy as np
from mediapipe.tasks.python import vision
import mediapipe as mp
from tqdm.auto import tqdm
import os

class FaceCropExtractionPipeline:
    def __init__(self, face_detector_model_path: str, min_detection_confidence: float,
                 bbox_margin_frac: float, sampling_fps: int = 5, max_frames_per_clip: int = None):
        # Stores the model *path*, not a live MediaPipe BaseOptions object -- see the
        # ProcessPoolExecutor note in the extraction loop below for why that distinction matters.
        self.face_detector_model_path = face_detector_model_path
        self.min_detection_confidence = min_detection_confidence
        self.bbox_margin_frac = bbox_margin_frac
        self.sampling_fps = sampling_fps
        self.max_frames_per_clip = max_frames_per_clip  # None = no cap

    def _expand_and_clip_bbox(self, x, y, w, h, frame_w, frame_h):
        mx, my = int(w * self.bbox_margin_frac), int(h * self.bbox_margin_frac)
        x0 = max(0, x - mx)
        y0 = max(0, y - my)
        x1 = min(frame_w, x + w + mx)
        y1 = min(frame_h, y + h + my)
        return x0, y0, x1, y1

    def process_video(self, video_path: str):
        """Returns a list of (frame_idx, sample_idx, cropped_bgr_image) for every frame with a confident
        face detection. Crops are variable-size -- resizing to a fixed input shape is left to
        07_cnn_training.ipynb, not baked in here, so the raw crops stay inspectable as-is."""
        from mediapipe.tasks import python as mp_python
        # BaseOptions is built fresh here, inside whatever process actually calls process_video
        # (the worker, once the fix below is in place) -- never passed in from outside.
        base_options = mp_python.BaseOptions(model_asset_path=self.face_detector_model_path)
        options = vision.FaceDetectorOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_detection_confidence=self.min_detection_confidence,
        )
        detector_local = vision.FaceDetector.create_from_options(options)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            detector_local.close()
            raise IOError(f"Cannot open video: {video_path}")

        src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_stride = max(1, round(src_fps / self.sampling_fps))

        crops = []
        frame_idx = 0
        sample_idx = 0  # increments once per SAMPLED frame (every frame_stride-th), regardless
                         # of whether a face was detected -- this is the consecutive-integer
                         # position a downstream windowing notebook needs; frame_idx alone is
                         # spaced by frame_stride and has no such guarantee.
        pbar = tqdm(total=total_frames, desc=f"Processing {os.path.basename(video_path)[:20]}...", leave=False)

        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            if frame_idx % frame_stride == 0:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                timestamp_ms = int(frame_idx * (1000.0 / src_fps))

                result = detector_local.detect_for_video(mp_image, timestamp_ms)

                if result.detections:
                    # Take the highest-confidence detection if more than one face is found.
                    best = max(result.detections, key=lambda d: d.categories[0].score)
                    bbox = best.bounding_box
                    frame_h, frame_w = frame_bgr.shape[:2]
                    x0, y0, x1, y1 = self._expand_and_clip_bbox(
                        bbox.origin_x, bbox.origin_y, bbox.width, bbox.height, frame_w, frame_h
                    )
                    if x1 > x0 and y1 > y0:
                        # Crop from the original BGR frame -- cv2.imwrite expects BGR and
                        # handles the JPEG color-space conversion internally, so this round-trips
                        # correctly with cv2.imread AND with standard decoders like
                        # tf.io.decode_jpeg (both expect/produce a normal RGB-ordered JPEG file).
                        crop_bgr = frame_bgr[y0:y1, x0:x1].copy()
                        crops.append((frame_idx, sample_idx, crop_bgr))

                sample_idx += 1

                if self.max_frames_per_clip is not None and len(crops) >= self.max_frames_per_clip:
                    # Hard cap reached -- stop sampling this clip early rather than reading (and
                    # discarding) the rest of the video. A simple ceiling, not an
                    # adaptive/similarity-based sampler.
                    break

            frame_idx += 1
            pbar.update(1)

        pbar.close()
        cap.release()
        detector_local.close()
        return crops


### Instantiate Components and Resume/Reset Option

A full extraction pass over every clip can take hours, so this is resumable by default: if
`face_crops_index.csv` already exists, clips already recorded in it are skipped. Newly processed
crops are **appended to the CSV the moment each image is written** to disk (not accumulated in
memory and written once per video, or once at the end) -- so an interrupted or crashed run
doesn't lose completed work, even work partway through a single video, and a re-run picks up
wherever it left off rather than starting over. Because extraction runs across multiple worker
processes in parallel (see the Video Processing Loop below), every append is guarded by a
`multiprocessing.Manager().Lock()` shared across workers, so two workers finishing a crop at the
same instant can't interleave or corrupt the CSV, or both think they're the first writer and
duplicate the header row.


In [ ]:
import pandas as pd

# Note: this pipeline object holds only a model *path* string + floats/ints (see the class
# definition above) -- no live MediaPipe object -- so, unlike 01_dataset_creation_lstm.ipynb's
# equivalent pipeline (which has to hold a live TF layer + FaceLandmarkerOptions and therefore
# needs a ProcessPoolExecutor(initializer=...) to avoid pickling those across the process
# boundary), it's safe to pass this one straight into executor.submit() below: pickling a
# handful of primitives can't corrupt native state the way pickling a live TF/MediaPipe object
# can, which is what actually caused the BrokenProcessPool crash in that other notebook.
face_crop_pipeline = FaceCropExtractionPipeline(
    face_detector_model_path, min_detection_confidence, bbox_margin_frac,
    sampling_fps=sampling_fps, max_frames_per_clip=MAX_FRAMES_PER_CLIP,
)

face_crops_index_csv_path = os.path.join(processed_folder, "face_crops_index.csv")

# --- Resume vs. full-reset flag ---
# False (default): RESUME mode. Clips already recorded in an existing face_crops_index.csv are
# skipped; newly processed crops are appended to it as their image is written. Safe to
# interrupt/re-run without redoing hours of already-completed work.
# True: wipe face_crops_folder and face_crops_index.csv and rebuild everything from zero -- use
# this only when you deliberately want a full, clean rebuild (e.g. after changing
# bbox_margin_frac or min_detection_confidence, which would make old crops inconsistent with new
# ones).
reset_dataset = True

if reset_dataset:
    import shutil
    if os.path.exists(face_crops_folder):
        shutil.rmtree(face_crops_folder)
        print(f"🧹 Face crops directory removed: {face_crops_folder}")
    if os.path.exists(face_crops_index_csv_path):
        os.remove(face_crops_index_csv_path)
        print(f"🧹 Index CSV removed: {face_crops_index_csv_path}")
    os.makedirs(face_crops_folder, exist_ok=True)
    already_done = set()
    print("✨ Ready for fresh extraction (reset_dataset=True -- starting from zero).")
else:
    os.makedirs(face_crops_folder, exist_ok=True)
    if os.path.exists(face_crops_index_csv_path):
        existing_df = pd.read_csv(face_crops_index_csv_path)
        already_done = set(zip(existing_df['subject'], existing_df['parent_video']))
        print(f"📦 Resuming: {len(already_done)} clip(s) already recorded in "
              f"{face_crops_index_csv_path}, will be skipped.")
    else:
        already_done = set()
        print("No existing index CSV found -- starting fresh (resume mode, nothing to skip yet).")


### Recovering an Index from Already-Cropped Files (Optional)

Run this cell **only if** `face_crops_folder` already has `.jpg` files in it from an earlier run
that never reached the "Index CSV Summary" step below — it rebuilds `face_crops_index.csv` by
scanning those files directly (parsing `{subject}_{clip_stem}_s{sample_idx}.jpg` filenames)
instead of re-running the (potentially many-hour) extraction pass over video. After this runs,
the resume-aware extraction loop below will correctly see those clips as already done and only
process whatever's genuinely still missing.


In [ ]:
import glob
import re
import os
import cv2
import pandas as pd
from tqdm.auto import tqdm

def rebuild_index_from_existing_crops(face_crops_folder):
    pattern = re.compile(r'^(subject_\d+)_(.+)_s(\d+)\.jpg$')

    image_paths = sorted(glob.glob(os.path.join(face_crops_folder, "*.jpg")))
    rows = []
    n_unrecognized = 0
    n_skipped = 0

    for image_path in tqdm(image_paths, desc="Scanning existing crops"):
        fname = os.path.basename(image_path)
        m = pattern.match(fname)
        if not m:
            print(f"⚠️  Skipping unrecognized filename: {fname}")
            n_unrecognized += 1
            continue
        subject, clip_stem, sample_idx_str = m.groups()

        level_match = re.search(r'level_(\d+)', clip_stem, re.IGNORECASE)
        if not level_match:
            print(f"⚠️  Skipping {subject}_{clip_stem}: couldn't parse a level from the clip name")
            n_skipped += 1
            continue
        raw_level = int(level_match.group(1))
        try:
            level = map_level(raw_level)
        except ValueError as e_val:
            print(f"⚠️  Skipping {subject}_{clip_stem}: {e_val}")
            n_skipped += 1
            continue

        img = cv2.imread(image_path)
        if img is None:
            # Most likely a truncated/corrupt file from a crash mid-write.
            print(f"⚠️  Skipping {fname}: image failed to load (likely truncated)")
            n_skipped += 1
            continue
        h, w = img.shape[:2]

        rows.append({
            "subject": subject,
            "level": level,
            "parent_video": f"{clip_stem}.mp4",
            "frame_idx": None,
            "sample_idx": int(sample_idx_str),
            "image_path": image_path,
            "crop_width": w,
            "crop_height": h,
        })

    if n_unrecognized or n_skipped:
        print(f"\n⚠️  {n_unrecognized} unrecognized filename(s), {n_skipped} unloadable/unparsable file(s) skipped.")
    return pd.DataFrame(rows)

df_recovered = rebuild_index_from_existing_crops(face_crops_folder)

if len(df_recovered) == 0:
    print("No existing crop files found (or none matched the expected filename pattern) -- "
          "nothing to recover. Proceed to the normal extraction loop below.")
else:
    df_recovered = df_recovered.sort_values(['subject', 'parent_video', 'sample_idx']).reset_index(drop=True)
    df_recovered.to_csv(face_crops_index_csv_path, index=False)
    print(f"✅ Recovered index CSV from {len(df_recovered)} existing crop files.")
    print(f"   Clips represented: {df_recovered.groupby(['subject', 'parent_video']).ngroups}")
    print(f"   Written to: {face_crops_index_csv_path}")


### Video Processing Loop

For each **not-yet-done** video: read the subject and the class label from its path/filename
(`map_level` validates the `level_` number — see the config note above), run
`face_crop_pipeline.process_video`, and write every returned crop as its own `.jpg` file under
`face_crops/`, named `{subject}_{clip_stem}_s{sample_idx}.jpg`. All crops land in one flat
directory — the label lives in the index CSV, not in a class-subfolder path, so subject-grouped
splitting stays possible in the training notebook. Each crop's row is appended to the CSV
**immediately after its image is written**, guarded by a shared lock across the parallel
workers -- not batched per video or written once at the end -- see the resume note above.


In [ ]:
import re
import cv2
import os
import multiprocessing
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_single_video_crops(video_path, pipeline_instance, face_crops_folder, jpeg_quality,
                                csv_path, csv_lock):
    filename = os.path.basename(video_path)
    subject = os.path.basename(os.path.dirname(video_path))
    clip_stem = os.path.splitext(filename)[0]

    level_match = re.search(r'level_(\d+)', filename, re.IGNORECASE)
    if not level_match:
        return None, (filename, "Missing 'level_' prefix")

    raw_level = int(level_match.group(1))
    try:
        level = map_level(raw_level)
    except ValueError as e:
        return None, (filename, str(e))

    try:
        crops = pipeline_instance.process_video(video_path)
    except IOError as e:
        return None, (filename, str(e))

    if not crops:
        return None, (filename, "No confident face detections in any sampled frame")

    local_rows = []
    for frame_idx, sample_idx, crop_bgr in crops:
        # Filename keyed on sample_idx (the consecutive-position counter), not frame_idx (the raw
        # video frame count spaced by frame_stride) -- sample_idx is what downstream windowing
        # (09_dataset_creation_cnn_lstm.ipynb) actually needs to detect gaps correctly.
        image_filename = f"{subject}_{clip_stem}_s{sample_idx}.jpg"
        image_path = os.path.join(face_crops_folder, image_filename)
        cv2.imwrite(image_path, crop_bgr, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        h, w = crop_bgr.shape[:2]
        row = {
            "subject": subject,
            "level": level,
            "parent_video": filename,
            "frame_idx": frame_idx,
            "sample_idx": sample_idx,
            "image_path": image_path,
            "crop_width": w,
            "crop_height": h,
        }
        # Write this row the moment its image is saved, not after the whole video finishes --
        # a crash mid-video no longer loses the crops it already wrote. Guarded by a lock
        # shared across all worker processes: two workers can otherwise interleave rows or,
        # worse, both see "file doesn't exist yet" and each write their own header line.
        with csv_lock:
            write_header = not os.path.exists(csv_path)
            pd.DataFrame([row]).to_csv(csv_path, mode='a', header=write_header, index=False)
        local_rows.append(row)
    return local_rows, None

def _video_key(video_path):
    filename = os.path.basename(video_path)
    subject = os.path.basename(os.path.dirname(video_path))
    return (subject, filename)

videos_to_process = [vp for vp in video_files if _video_key(vp) not in already_done]
print(f"{len(videos_to_process)} of {len(video_files)} clip(s) need processing "
      f"({len(video_files) - len(videos_to_process)} already done, will be skipped).")

skipped = []
total_new_rows = 0
max_workers = os.cpu_count() or 2

if videos_to_process:
    print(f"🚀 Starting parallel extraction with {max_workers} workers...")

    # A Manager().Lock() (not a plain multiprocessing.Lock()) so it can be passed cleanly into
    # executor.submit() and shared by every worker, serializing their CSV appends below.
    manager = multiprocessing.Manager()
    csv_lock = manager.Lock()

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(
                process_single_video_crops, vp, face_crop_pipeline, face_crops_folder, jpeg_quality,
                face_crops_index_csv_path, csv_lock,
            ) for vp in videos_to_process
        ]
        main_pbar = tqdm(as_completed(futures), total=len(futures), desc="Parallel Progress")
        for future in main_pbar:
            local_rows, error = future.result()
            if error:
                skipped.append(error)
                continue
            if local_rows:
                # Rows for this video are already on disk -- written one-by-one, as each crop
                # was captured, inside process_single_video_crops above. Just tally the count.
                total_new_rows += len(local_rows)

    print(f"\nExtraction finished this run: {total_new_rows} new face crops added.")
    if skipped:
        print(f"Skipped {len(skipped)} clip(s) due to errors:")
        for name, reason in skipped:
            print(f"   - {name}: {reason}")
else:
    print("Nothing to process -- every clip is already in the index CSV.")


### Dataset Summary & Completeness Check

Reads the finished (or resumed) `face_crops_index.csv` back and reports whether extraction
actually succeeded: totals, any clips skipped this run, per-class counts, and — the check that
actually matters for training — per-subject class coverage against what's really in
`raw_videos_binary/`, not just whether a class exists *somewhere* in the dataset. Also previews one
crop per class.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if not os.path.exists(face_crops_index_csv_path):
    raise Exception(f"⚠️ No index CSV exists yet at '{face_crops_index_csv_path}' -- nothing has "
                     "been successfully processed or recovered.")

df_face_crops = pd.read_csv(face_crops_index_csv_path)
print(f"✅ Index CSV: {len(df_face_crops)} total face crops across "
      f"{df_face_crops.groupby(['subject', 'parent_video']).ngroups} clips.")
print(f"   {face_crops_index_csv_path}")
if skipped:
    print(f"⚠️  {len(skipped)} clip(s) skipped this run due to errors:")
    for name, reason in skipped:
        print(f"   - {name}: {reason}")

level_counts = df_face_crops['level'].value_counts().sort_index()
print(f"\n{NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"Distinct subjects: {df_face_crops['subject'].nunique()}")
print(f"Crop size range: {df_face_crops['crop_width'].min()}-{df_face_crops['crop_width'].max()} (w) x "
      f"{df_face_crops['crop_height'].min()}-{df_face_crops['crop_height'].max()} (h)")
print("\nCrop counts per class:")
for lvl, count in level_counts.items():
    name = CLASS_NAMES[int(lvl) - 1] if 1 <= int(lvl) <= NUM_CLASSES else "?"
    print(f"  Level {lvl} ({name}): {count} crops")

# --- Per-subject completeness check ---
# The count above is easy to satisfy while still being badly incomplete: it only asks "does class
# X exist anywhere," not "does every subject have it," and it says nothing about whether every
# subject with raw video actually got processed. This check is against `video_files` (the actual
# raw video tree contents from earlier in this notebook), not a hardcoded subject count, so it
# stays correct as more subjects are added.
raw_subjects = sorted({os.path.basename(os.path.dirname(vp)) for vp in video_files},
                       key=lambda s: int(s.split('_')[-1]))
indexed_subjects = set(df_face_crops['subject'].unique())

print(f"\n--- Per-Subject Completeness ({len(raw_subjects)} subject(s) found in raw_videos_binary/) ---")
incomplete_subjects = []
for subj in raw_subjects:
    if subj not in indexed_subjects:
        print(f"  ❌ {subj}: no crops at all yet (not processed, or every clip failed/was skipped)")
        incomplete_subjects.append(subj)
        continue
    subj_levels = set(df_face_crops.loc[df_face_crops['subject'] == subj, 'level'])
    missing = [CLASS_NAMES[l - 1] for l in range(1, NUM_CLASSES + 1) if l not in subj_levels]
    if missing:
        print(f"  ⚠️  {subj}: missing {missing}")
        incomplete_subjects.append(subj)
    else:
        print(f"  ✅ {subj}: all {NUM_CLASSES} classes present")

if incomplete_subjects:
    print(f"\n⚠️ {len(incomplete_subjects)}/{len(raw_subjects)} subject(s) incomplete: {incomplete_subjects}")
    print("   Re-run the Video Processing Loop above (resume mode, reset_dataset=False) to fill "
          "these in -- it only processes videos not yet in the index, so this is safe/cheap to "
          "re-run. If a subject stays incomplete after a full re-run, that subject's clip(s) "
          f"for the missing class(es) aren't in raw_videos_binary/ -- re-run relabel_binary_raw_videos.ipynb "
          f"if you've added footage to raw_videos/, else check Drive directly.")
else:
    print(f"\n✅ Success: all {len(raw_subjects)} subject(s) in raw_videos_binary/ have all {NUM_CLASSES} classes represented.")

# Preview a handful of crops, one per class where available
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4 * NUM_CLASSES, 4))
for ax, lvl in zip(axes, range(1, NUM_CLASSES + 1)):
    sample = df_face_crops[df_face_crops['level'] == lvl].sample(1, random_state=0) if lvl in level_counts.index else None
    if sample is not None:
        img = mpimg.imread(sample.iloc[0]['image_path'])
        ax.imshow(img)
    ax.set_title(CLASS_NAMES[lvl - 1])
    ax.axis('off')
plt.tight_layout()
plt.show()
